In [ ]:
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import math
from collections import defaultdict

DATA_DIR = "./data"

# ── Load embeddings ────────────────────────────────────────────────────────────
with open(f"{DATA_DIR}/embeddings/genre_embeddings.json") as f:
    genre_raw = json.load(f)

with open(f"{DATA_DIR}/embeddings/tag_embeddings.json") as f:
    tag_raw = json.load(f)

genres    = list(genre_raw.keys())
token2idx = {g: i for i, g in enumerate(genres)}
Z         = np.array(list(genre_raw.values()), dtype=np.float32)
Z_norm    = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

tags_vocab   = list(tag_raw.keys())
tag2idx      = {t: i for i, t in enumerate(tags_vocab)}
Z_tags       = np.array(list(tag_raw.values()), dtype=np.float32)
Z_tags_norm  = Z_tags / (np.linalg.norm(Z_tags, axis=1, keepdims=True) + 1e-9)

# ── Anchor colors ──────────────────────────────────────────────────────────────
ANCHORS = {
    "death metal":    np.array([20,  10,  30],  dtype=np.float32),
    "black metal":    np.array([10,  10,  10],  dtype=np.float32),
    "pop":            np.array([255, 105, 180],  dtype=np.float32),
    "jazz":           np.array([180, 120, 40],   dtype=np.float32),
    "classical":      np.array([200, 200, 255],  dtype=np.float32),
    "edm":            np.array([0,   255, 200],  dtype=np.float32),
    "tropical house": np.array([255, 200, 50],   dtype=np.float32),
    "blues":          np.array([30,  80,  180],  dtype=np.float32),
    "folk":           np.array([120, 180, 80],   dtype=np.float32),
    "reggaeton":      np.array([255, 80,  50],   dtype=np.float32),
    "house":          np.array([100, 0,   255],  dtype=np.float32),
    "rap":            np.array([50,  50,  50],   dtype=np.float32),
    "country":        np.array([210, 140, 50],   dtype=np.float32),
    "r&b":            np.array([150, 0,   100],  dtype=np.float32),
    "soul":           np.array([200, 80,  20],   dtype=np.float32),
}

anchor_keys   = [k for k in ANCHORS if k in token2idx]
anchor_idx    = np.array([token2idx[k] for k in anchor_keys])
anchor_vecs   = Z_norm[anchor_idx]
anchor_colors = np.array([ANCHORS[k] for k in anchor_keys])

print(f"Loaded {len(genres)} genres, {len(tags_vocab)} tags, {len(anchor_keys)}/{len(ANCHORS)} anchors found")

# ── Token → RGB ────────────────────────────────────────────────────────────────
def genre_to_rgb(token):
    token = token.lower().strip()
    if token not in token2idx:
        return None
    q       = Z_norm[token2idx[token]]
    sims    = anchor_vecs @ q
    sims    = sims - sims.max()
    weights = np.exp(sims * 5)
    weights /= weights.sum()
    return np.clip((weights[:, None] * anchor_colors).sum(0), 0, 255).astype(np.float32)

# ── Song → RGB (genres weighted most, tags least) ─────────────────────────────
FIELD_WEIGHTS = {"genres": 1.0, "tags": 0.3}

def song_to_rgb(song):
    total_rgb, total_w = np.zeros(3, dtype=np.float32), 0.0
    for genre in song.get("genres", []):
        rgb = genre_to_rgb(genre)
        if rgb is not None:
            total_rgb += FIELD_WEIGHTS["genres"] * rgb
            total_w   += FIELD_WEIGHTS["genres"]
    for tag in song.get("tags", []):
        t = tag.lower().strip()
        if t in tag2idx:
            q       = Z_tags_norm[tag2idx[t]]
            sims    = anchor_vecs @ q
            sims    = sims - sims.max()
            weights = np.exp(sims * 5)
            weights /= weights.sum()
            rgb     = np.clip((weights[:, None] * anchor_colors).sum(0), 0, 255)
            total_rgb += FIELD_WEIGHTS["tags"] * rgb
            total_w   += FIELD_WEIGHTS["tags"]
    if total_w == 0:
        return np.array([128, 128, 128], dtype=np.uint8)
    return np.clip(total_rgb / total_w, 0, 255).astype(np.uint8)

# ── Generate blob ──────────────────────────────────────────────────────────────
def generate_blob(song, size=300):
    rgb     = song_to_rgb(song)
    r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])

    cx, cy  = size // 2, size // 2
    radius  = size // 2.5

    y_grid, x_grid = np.mgrid[0:size, 0:size]
    dist    = np.sqrt((x_grid - cx)**2 + (y_grid - cy)**2)
    t       = np.clip(dist / radius, 0, 1)
    falloff = 1 - t

    bg = np.array([15, 15, 15])
    fg = np.array([r, g, b])

    pixel_r = (fg[0] * falloff + bg[0] * (1 - falloff)).astype(np.uint8)
    pixel_g = (fg[1] * falloff + bg[1] * (1 - falloff)).astype(np.uint8)
    pixel_b = (fg[2] * falloff + bg[2] * (1 - falloff)).astype(np.uint8)

    img = Image.fromarray(np.stack([pixel_r, pixel_g, pixel_b], axis=2))
    return img, rgb

# ── Load songs + tags for spotify lookup ──────────────────────────────────────
def read_csv(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        reader = __import__('csv').reader(f, delimiter=';')
        raw_headers = next(reader)
        headers = [h.strip().strip('"') for h in raw_headers[0].split(',')]
        return [dict(zip(headers, [p.strip().strip('"') for p in row])) for row in reader if len(row) == len(headers)]

songs_rows = read_csv(f"{DATA_DIR}/csv/songs.csv")
tags_rows  = read_csv(f"{DATA_DIR}/csv/tags.csv")

song_lookup  = {row['spotify_id']: row for row in songs_rows}
song_genres  = defaultdict(set)
for row in songs_rows:
    song_genres[row['spotify_id']].add(row['genre_name'].lower())

song_tags = defaultdict(list)
for row in tags_rows:
    sid = row['song_spotify_id']
    tag = row['tag'].lower().strip()
    pop = int(row['popularity']) if row['popularity'].isdigit() else 0
    song_tags[sid].append((tag, pop))
for sid in song_tags:
    song_tags[sid] = [t for t, p in sorted(song_tags[sid], key=lambda x: -x[1])[:5]]

print(f"Loaded {len(song_lookup):,} songs")

# ── Spotify ID → blob ──────────────────────────────────────────────────────────
def blob_from_spotify_id(spotify_id, size=300):
    if spotify_id not in song_genres:
        print(f"'{spotify_id}' not found")
        return None, None
    song = {
        "genres": list(song_genres[spotify_id]),
        "tags":   song_tags.get(spotify_id, [])
    }
    return generate_blob(song, size=size)

# ── Plot ───────────────────────────────────────────────────────────────────────
TEST_IDS = [
    "6lV2MSQmRIkycDScNtrBXO",
    "3CorimZuPQslcTZop2wp2Z",
    "4tNmBj1etOHrCN7otGjweK",
    "2fRAzmKCcnS9c1sAZd5uUM",
    "4zlj8rTf9XiKqZ1nAwF9Bj",
    "3Izog7LLV9zViAsBZaOJ5t",
    "3JJA96pbxukWALeKoBIL4i",
    "6VNE73gx82omH0XWSiq3FW",
]

n_cols = 4
n_rows = math.ceil(len(TEST_IDS) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 6 * n_rows))
fig.patch.set_facecolor('#0f0f0f')
axes = axes.flatten()

for i, sid in enumerate(TEST_IDS):
    img, rgb = blob_from_spotify_id(sid)
    row      = song_lookup.get(sid, {})
    genres_str = ", ".join(song_genres.get(sid, []))
    tags_str   = ", ".join(song_tags.get(sid, [])[:3])

    if img is not None:
        axes[i].imshow(img)
        axes[i].set_facecolor('#0f0f0f')
        axes[i].set_title(
            row.get('name', sid),
            fontsize=9, fontweight='bold', color='white', pad=6
        )
        axes[i].set_xlabel(
            f"{row.get('artist', '')}\n🎵 {genres_str}\n🏷  {tags_str}\nRGB{tuple(rgb)}",
            fontsize=7, color='#cccccc', labelpad=8
        )
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        for spine in axes[i].spines.values():
            spine.set_visible(False)

for j in range(len(TEST_IDS), len(axes)):
    axes[j].axis('off')

plt.suptitle("Song → Color Blob", fontsize=16, color='white', y=1.01)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/sonblobs.png", dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
